In [1]:
import numpy as np
import tensorflow as tf
import keras
import matplotlib.pyplot as plt

import sys

sys.path.append("../")

In [2]:
import numpy as np

seq_length = 40
dim = 2
noise = 0


def sample_circle(n, max_n=seq_length, dim=2, noise=0.05):
    points = np.full((max_n, dim), -1, dtype=np.float32)
    sample_points = np.random.randn(n, dim)
    sample_points /= np.linalg.norm(sample_points, axis=1, keepdims=True)
    r = np.random.uniform(0.5, 1)
    sample_points *= r  # Scale by random radius
    if noise > 0:
        noise_points = np.random.normal(0, 1, sample_points.shape) * noise * r
        sample_points += noise_points
    points[:n, :] = sample_points
    return points


def sample_square(n, max_n=seq_length, dim=2, noise=0.05):
    points = np.full((max_n, dim), -1, dtype=np.float32)
    sample_points = np.random.uniform(-1, 1, (n, dim))
    sample_points /= np.linalg.norm(sample_points, axis=1, keepdims=True, ord=np.inf)
    r = np.random.uniform(0.5, 1)
    sample_points *= r  # Scale by random radius
    if noise > 0:
        noise_points = np.random.normal(0, 1, sample_points.shape) * noise * r
        sample_points += noise_points
    points[:n, :] = sample_points
    return points


dataset_size = 10000
X_sphere = np.zeros((dataset_size, seq_length, dim), dtype=np.float32)
X_square = np.zeros((dataset_size, seq_length, dim), dtype=np.float32)

for iter in range(dataset_size):
    n = seq_length
    X_sphere[iter, :, :] = sample_circle(n, seq_length, dim, noise)
    X_square[iter, :, :] = sample_square(n, seq_length, dim, noise)

X = np.concatenate([X_sphere, X_square], axis=0)
y = np.concatenate(
    [
        np.zeros((dataset_size, 1), dtype=np.float32),
        np.ones((dataset_size, 1), dtype=np.float32),
    ],
    axis=0,
)

np.random.seed(42)
shuffle_indices = np.random.permutation(X.shape[0])
X = X[shuffle_indices]
y = y[shuffle_indices]

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
from src.model.components import (
    MultiHeadAttentionBlock,
    MultiHeadAttentionStack,
    SelfAttentionBlock,
    SelfAttentionStack,
    PoolingAttentionBlock,
    point_transformer,
    GenerateMask,
    GetSequenceLength,
    DecoderQueries,
    GenerateDecoderMask,
    MLP,
    MaskOutput,
    DecoderPoints,
)

input_dim = dim
feature_dim = 8
latent_dim = 32
num_seeds = latent_dim // feature_dim
dropout_rate = 0.0
num_heads = 8
regularizer = None

input = keras.layers.Input(shape=(seq_length, input_dim), name="input")

input_embedding = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="pixel_embedding",
    activation="relu",
    regularizer=regularizer,
)(input)

pixel_self_attention = SelfAttentionStack(
    num_heads=num_heads,
    key_dim=feature_dim,
    stack_size=3,
    name="pixel_self_attention",
    regularizer=regularizer,
)(input_embedding)

pooling = PoolingAttentionBlock(
    key_dim=feature_dim,
    num_seeds=num_seeds,
    num_heads=num_heads,
    name="pixel_pooling",
)(pixel_self_attention)


pooling_self_attention = SelfAttentionBlock(
    num_heads=num_heads,
    key_dim=feature_dim,
    name="pooling_self_attention",
    regularizer=regularizer,
)(pooling)

latent_space = MLP(
    num_layers=2,
    output_dim=feature_dim,
    name="latent_space",
    activation="linear",
    regularizer=regularizer,
    dropout_rate=dropout_rate,
)(pooling_self_attention)

latent_output = keras.layers.Flatten(name="latent_output")(latent_space)

decoder_points = DecoderQueries(
    num_queries=seq_length,
    feature_dim=feature_dim,
    regularizer=regularizer,
)(latent_space)


latent_queries = latent_space


decoded_output = MultiHeadAttentionBlock(
    num_heads=num_heads,
    key_dim=feature_dim,
    name="decoded_output_attention_1",
    regularizer=regularizer,
)(decoder_points, latent_queries)

decoded_output = MultiHeadAttentionBlock(
    num_heads=num_heads,
    key_dim=feature_dim,
    name="decoded_output_attention_2",
    dropout_rate=dropout_rate,
    regularizer=regularizer,
)(decoded_output,latent_queries)


decoded_output = SelfAttentionStack(
    num_heads=num_heads,
    key_dim=feature_dim,
    stack_size=2,
    dropout_rate=dropout_rate,
    name="decoded_output_self_attention",
    regularizer=regularizer,
)(decoded_output)

decoded_output = MLP(
    num_layers=2,
    output_dim=input_dim,
    name="decoded_output",
    activation="linear",
)(decoded_output)


"""
seqlen_output = keras.layers.Concatenate(name="seqlen_output")(
    [
        keras.layers.Flatten(name="seqlen_flatten")(pixel_seqlen),
        keras.layers.Flatten(name="decoder_seqlen_flatten")(decoder_seqlen),
    ]
)"""

SetAE = keras.Model(
    inputs=input,
    outputs=[decoded_output, latent_output],
    name="SetAutoEncoder",
)

In [5]:
Encoder = keras.Model(
    inputs=input,
    outputs=latent_output,
    name="Encoder",
)

In [ ]:
from src.training import (
    SplitMSE,
    ChamferDistanceMasked,
    EmbeddingSpaceSpreading,
    ChamferDistance,
    EmbeddingSpaceSpreading,
    EarthMoversDistanceLoss,
)

SetAE.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        "decoded_output": EarthMoversDistanceLoss(),
        "latent_output": EmbeddingSpaceSpreading(),
    },
    loss_weights={"decoded_output": 1.0, "latent_output": 0.1},
)

SetAE.summary()
SetAE.fit(
    X,
    [X, np.zeros((X.shape[0], latent_dim), dtype=np.float32)],
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    ],
)

Model: "SetAutoEncoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 40, 2)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_embedding     │ (None, 40, 8)     │        182 │ input[0][0]       │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_self_attenti… │ (None, 40, 8)     │      7,680 │ pixel_embedding[… │
│ (SelfAttentionStac… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_pooling       │ (None, 4, 8)      │      2,592 │ pixel_self_atten… │
│ (PoolingAttentionB… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pooling_self_atten… │ (None, 4, 8)      │      2,560 │ pixel_pooling[0]… │
│ (SelfAttentionBloc… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latent_space (MLP)  │ (None, 4, 8)      │        288 │ pooling_self_att… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_queries     │ (None, 40, 8)     │        320 │ latent_space[0][… │
│ (DecoderQueries)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_output_att… │ (None, 40, 8)     │      2,560 │ decoder_queries[… │
│ (MultiHeadAttentio… │                   │            │ latent_space[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_output_att… │ (None, 40, 8)     │      2,560 │ decoded_output_a… │
│ (MultiHeadAttentio… │                   │            │ latent_space[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_output_sel… │ (None, 40, 8)     │      5,120 │ decoded_output_a… │
│ (SelfAttentionStac… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_output      │ (None, 40, 2)     │         92 │ decoded_output_s… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latent_output       │ (None, 32)        │          0 │ latent_space[0][… │
│ (Flatten)           │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,953 (93.57 KB)

 Trainable params: 23,953 (93.57 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 85s 133ms/step - decoded_output_loss: 0.6150 - latent_output_loss: 11.6700 - loss: 1.7820 - val_decoded_output_loss: 0.5404 - val_latent_output_loss: 8.7649 - val_loss: 1.4168 - learning_rate: 1.0000e-04
Epoch 2/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 69s 139ms/step - decoded_output_loss: 0.5184 - latent_output_loss: 8.2860 - loss: 1.3470 - val_decoded_output_loss: 0.4927 - val_latent_output_loss: 7.6901 - val_loss: 1.2617 - learning_rate: 1.0000e-04
Epoch 3/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - decoded_output_loss: 0.4831 - latent_output_loss: 7.5740 - loss: 1.2405

In [ ]:
X_encoded_train = Encoder.predict(X_train).squeeze()
X_encoded_test = Encoder.predict(X_test).squeeze()

In [ ]:
X_encoded_train = (X_encoded_train - np.mean(X_encoded_train, axis=0)) / np.std(
    X_encoded_train, axis=0
)

In [ ]:
classifier_input = keras.layers.Input(shape=(latent_dim,), name="classifier_input")
classifier_mlp = MLP(
    num_layers=5,
    output_dim=1,
    name="classifier_mlp",
    activation="sigmoid",
)(classifier_input)

classifier = keras.Model(
    inputs=classifier_input,
    outputs=classifier_mlp,
    name="classifier",
)
classifier.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-2),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
classifier.fit(
    X_encoded_train,
    y_train,
    epochs=1000,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=100, restore_best_weights=True
        ),
    ],
    class_weight={label: np.mean(y_train == label) for label in np.unique(y_train)},
)

In [ ]:
event = 3
sample = X_sphere[event : event + 1]
decoded_output = SetAE.predict(sample)[0]
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(sample[0, :, 0], sample[0, :, 1], label="Input", color="blue")
ax.scatter(
    decoded_output[0, :, 0],
    decoded_output[0, :, 1],
    label="Decoded Output",
    color="red",
)
ax.set_title("Input vs Decoded Output")
ax.set_xlabel("X-axis")
ax.set_ylabel("Y-axis")
ax.legend()
plt.show()

In [ ]:
event = 0
sample = X_square[event : event + 1]
decoded_output = SetAE.predict(sample)[0]
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(sample[0, :, 0], sample[0, :, 1], label="Input", color="blue")
ax.scatter(
    decoded_output[0, :, 0],
    decoded_output[0, :, 1],
    label="Decoded Output",
    color="red",
)
ax.set_title("Input vs Decoded Output")
ax.set_xlabel("X-axis")
ax.set_ylabel("Y-axis")
ax.legend()
plt.show()

In [ ]:
from src.model.components import (
    MultiHeadAttentionStack,
    SelfAttentionStack,
    PoolingAttentionBlock,
    GenerateMask,
    MLP,
    GetSequenceLength,
    DecoderQueries,
)

feature_dim = 6
num_heads = 4

input_dim = dim
feature_dim = 8
latent_dim = 32
num_seeds = latent_dim
dropout_rate = 0.0
num_heads = 8

pixel_input = keras.layers.Input(shape=(seq_length, input_dim), name="pixel_input")
pixel_mask = GenerateMask(name="pixel_mask")(pixel_input)
pixel_seqlen = GetSequenceLength(name="pixel_seqlen")(pixel_mask)

pixel_embedding = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="pixel_embedding",
    activation="relu",
)(pixel_input)

pixel_self_attention = SelfAttentionStack(
    num_heads=num_heads,
    key_dim=feature_dim,
    stack_size=3,
    name="pixel_self_attention",
)(pixel_embedding)

pixel_pooling = PoolingAttentionBlock(
    key_dim=feature_dim,
    num_seeds=num_seeds,
    num_heads=num_heads,
    name="pixel_pooling",
)(pixel_self_attention)

pixel_latent_space = MLP(
    num_layers=4,
    output_dim=1,
    name="pixel_latent_space",
)(pixel_pooling)

pixel_latent_output = keras.layers.Flatten(name="pixel_latent_output")(
    pixel_latent_space
)

output = MLP(
    num_layers=4,
    output_dim=1,
    name="pixel_latent_output_mlp",
    activation="sigmoid",
)(pixel_latent_output)

transformer_model = keras.Model(
    inputs=pixel_input,
    outputs=output,
    name="ClassificationModel",
)
transformer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy()],
)

In [ ]:
transformer_model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    ],
    class_weight={label: np.mean(y_train == label) for label in np.unique(y_train)},
    shuffle=True,
)

In [ ]:
latent_encoder = keras.Model(
    inputs=pixel_input,
    outputs=pixel_latent_output,
    name="LatentEncoder",
)
latent_encoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.MeanSquaredError(),
    metrics=[keras.metrics.MeanAbsoluteError()],
)
latent_space = latent_encoder.predict(X_test)